# 🏷️ Project 1: Baseline E-Commerce Price Regression (Llama-3.2-3B)

### 📌 Overview & Purpose
This notebook implements a standard Supervised Fine-Tuning (SFT) baseline that maps raw e-commerce metadata (`Title`, `Category`, `Summary`) directly to a scalar currency prediction (`$XX.XX`). 

### ⚙️ Architecture & Setup
* **Base Model:** `meta-llama/Llama-3.2-3B`
* **Technique:** QLoRA (4-bit NF4 Quantization) via Hugging Face `peft` and `trl.SFTTrainer`
* **Target Modules:** Attention projection layers (`q_proj`, `k_proj`, `v_proj`, `o_proj`)
* **Optimization:** Paged AdamW 32-bit with Cosine Annealing

### ⚠️ Characteristics & Limitations
While this baseline converges quickly on regression loss, it functions as an opaque "black box" that memorizes dataset distributions without explaining its valuation logic. See `02_production_qwen_cot_engine.ipynb` for the production Chain-of-Thought architecture.

------------
------------

# The Price is Right

## Training (Fine-Tuning LLM)

In [1]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
wandb_api_key = user_secrets.get_secret("WANDB_API_KEY")

In [2]:
# pip uninstall -y torch torchvision torchaudio transformers trl peft accelerate bitsandbytes

# pip install \
# torch==2.6.0 \
# transformers==4.52.4 \
# accelerate==1.8.1 \
# peft==0.15.2 \
# trl==0.18.2 \
# bitsandbytes==0.46.0 \
# datasets==3.6.0 \
# huggingface_hub==0.33.0

!pip install -q --upgrade bitsandbytes==0.48.2 trl==0.25.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 28.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 465.5/465.5 kB 28.3 MB/s eta 0:00:00


In [3]:
import torch
import transformers
import trl
import peft
import accelerate
import bitsandbytes as bnb

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("trl:", trl.__version__)
print("peft:", peft.__version__)
print("accelerate:", accelerate.__version__)
print("bitsandbytes:", bnb.__version__)

torch: 2.10.0+cu128
transformers: 5.0.0
trl: 0.25.1
peft: 0.19.1
accelerate: 1.13.0
bitsandbytes: 0.48.2


In [4]:
# !pip install -q bitsandbytes trl
!wget -q https://raw.githubusercontent.com/ed-donner/llm_engineering/main/week7/util.py -O util.py

In [5]:
import os # file/system paths
import re # string handling
import math # math operations
from tqdm import tqdm #execution progress bars
from huggingface_hub import login # to authenticate the HF_HUB for gated models like Llama3.2
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, set_seed, BitsAndBytesConfig
                        #model architecture    #textprocessor,                               # 4-bit/8-bit quantization
from datasets import load_dataset, Dataset, DatasetDict
import wandb #for experiment tracking
from peft import LoraConfig #Sets up LoRA adapters to fine-tune only a tiny fraction(~1%) of the model parameters
from trl import SFTTrainer, SFTConfig #SFT wrapper built on top of transformers.Trainer, optimized for causalLM
from datetime import datetime # dynamic run naming
import matplotlib.pyplot as plt # loss plotting

In [6]:
# Constants

BASE_MODEL = "meta-llama/Llama-3.2-3B"
PROJECT_NAME = "price"
HF_USER = "Subhrajyoti75" #your HF name

LITE_MODE = True
DATA_USER = "ed-donner"
DATASET_NAME = f"{DATA_USER}/items_prompts_lite" if LITE_MODE else f"{DATA_USER}/items_prompts_full"

RUN_NAME = f"{datetime.now():%Y-%m-%d_%H.%M.%S}" #formats a timestamped run label for clear distinction in local output folders and W&B logs.
if LITE_MODE:
    RUN_NAME += "-lite"
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"

# Hyperparameters - overall
EPOCHS = 1 if LITE_MODE else 3
BATCH_SIZE = 32 if LITE_MODE else 256
MAX_SEQUENCE_LENGTH = 128 # max-token length
GRADIENT_ACCUMULATION_STEPS = 1 #updates weights every single batch step since the micro-batch size already fits comfortably in VRAM

# Hyperparameters - QLoRA
QUANT_4_BIT = True
LORA_R = 32 if LITE_MODE else 256 #rank=32
LORA_ALPHA = LORA_R * 2 #scaling factor(alpha = 2 x 'r')
ATTENTION_LAYERS = ["q_proj", "v_proj", "k_proj", "o_proj"]
MLP_LAYERS = ["gate_proj", "up_proj", "down_proj"]
TARGET_MODULES = ATTENTION_LAYERS if LITE_MODE else ATTENTION_LAYERS + MLP_LAYERS #prevent overfitting in small dataset
LORA_DROPOUT = 0.1 #adds 10% dropout to LoRA layers to mitigate overfitting

# Hyperparameters - training
LEARNING_RATE = 1e-4
WARMUP_RATIO = 0.01 # the % of total training steps during which the LR gradually rramps up from 0 to max specified LR(1e-4)
LR_SCHEDULER_TYPE = 'cosine' # controls how lr changes over time after warmup period finishes.
WEIGHT_DECAY = 0.001 # a regularization step that subtracts a small fraction of weight values at each update step(L2 regularization)
OPTIMIZER = "paged_adamw_32bit" #memory efficient AdamW optimizer that offloads optimizer states to CPU RAM when VRAM spikes, preventing OOM

capability = torch.cuda.get_device_capability()
use_bf16 = capability[0] >= 8 #checks automatically GPU compatibility.If its 8.0 or higher(A100s) it enables bf16 else fp16

# Tracking
VAL_SIZE = 500 if LITE_MODE else 1000
LOG_STEPS = 5 if LITE_MODE else 10
SAVE_STEPS = 100 if LITE_MODE else 200
LOG_TO_WANDB = True

# warmup ratio is used as at the start of training, model adapters/gradients are randomly initialized or unbalanced. Applying a full high 
#     learining rate immediately can cause unstable gradient spikes, leading to loss divergence. Ramping up over 1%(0.01) of the steps 
#     stabilizes early optimization.

# in LR_SCHEDULER 'cosine' is used as with it the 'lr' follows a smooth cosine curve-decaying gradually toward nearly 0 by the final step.
#     This allows the model to make coarse weight adjustments early on and extremely fine adjustments near the end of training.

In [7]:
# A100 GPU supports this; T4 does not natively
use_bf16 

False

## Optimizers

The most common is Adam or AdamW(Adam with Weight Decay)

Adam achieves good convergence by storing the rolling average of the previous gradients; however, it adds an additional memory footprint of the order of the number of model parameters.

In [8]:
# Login to HuggingFace
os.environ["HF_TOKEN"] = hf_token
# login(hf_token)

# Login to Weights&Biases
os.environ["WANDB_API_KEY"] = wandb_api_key
wandb.login()

# Configure Weights & Biases to record against our project
os.environ["WANDB_PROJECT"] = PROJECT_NAME
os.environ["WANDB_LOG_MODEL"] = "false"
os.environ["WANDB_WATCH"] = "false"

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: subhrajyoti_soumyadarsan (subhrajyoti_soumyadarsan-independent-developer) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


### Why Wandb?

1. Live Loss and Metrics Tracking(Preventing Wasted Compute)

   Fine-tuning models(especially QLoRA/SFT) can take hours. W&B plots our `training loss, validation loss, and learining rate schedule` in real time. So if loss startit spiking(diverging) then we can step at step 50/2000 to save hours.

2. Kaggle Session Crash Protection

   If Kaggle environment disconencts, times out or run out of GPU memory(OOM), our local terminal logs and outputs are wiped or lost. Because SFTTrainer streams metrics directly to W&B cloud servers step-by-step, our execution history, plots and hyperparameter logs are permanently saved online.

3. Hyperparameter & Experiment Tracking

   When fine-tuning, we often tweak parameters like `learning_rate, lora_alpha, r, batch_size, or warmup_ratio`. W&B automatically logs all configs sent via `TrainingArgments` /`SFTConfig`, making it easy to compare multi-run charts to see which setup yielded the lowest evaluation loss.

4. Hardware Resource Monitoring

   W&B automatically records GPU VRAM usage, GPU utilization, CPU load, and system memory. This helps answer key operational questions:

    - Are you maxing out VRAM?
    - Is batch size or gradient_accumulation_steps optimal?
    - Is GPU utilization sitting at 99% or idling?

```python
training_args = SFTConfig(
    output_dir="./results",
    report_to="none",  # <--- Disables W&B entirely
    ...
)


In [9]:
dataset = load_dataset(DATASET_NAME)
train = dataset['train']
val = dataset['val'].select(range(VAL_SIZE)) #truncates validation size to 500 in LITE_MODE
test = dataset['test']

README.md:   0%|          | 0.00/509 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

data/val-00000-of-00001.parquet:   0%|          | 0.00/216k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/218k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [10]:
if LOG_TO_WANDB:
    wandb.init(project=PROJECT_NAME, name=RUN_NAME)
# explicitly starts a new run log under the W&B dashboard project using the timestamped run_name
#  it only initializes numerical metrics tracking

## Loading Tokenizer and Model

In [11]:
# Pick the right quantization
if QUANT_4_BIT:
    quant_config = BitsAndBytesConfig(
        load_in_4bit = True, #load base model in 4-bit quantization
        bnb_4bit_use_double_quant = True, # nested/double quantization of constants sacing ~0.4 bits per parameter
        # bnb_4bit_compute_dtype = torch.bfloat16 if use_bf16 else torch.float16, #dictates precision format for matmul
        bnb_4bit_compute_dtype=torch.float16,  # Force float16 here
        bnb_4bit_quant_type = "nf4" #Normalized float 4-bit
    )
else:
    quant_config = BitsAndBytesConfig(
        load_in_8bit = True, # load base model in 8-bit quantization
        bnb_8bit_compute_dtype = torch.float16 if use_bf16 else torch.float16, #outdated
    )

In [12]:
# Load the Tokenizer and the Model:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" #controls where padding tokens are placed

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config, #nf4 quantization 
    device_map="auto",#auto manages model across hardware
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id # synchronizes the model's text generation config with the newly 
# assigned tokenizer padding ID to suppress padding warnings during evaluation or generation.

print(f"Memory footprint: {base_model.get_memory_footprint() / 1e6:.1f} MB")

config.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

Memory footprint: 2197.6 MB


In [13]:
# # Enforce float16 on all model parameters
# for param in base_model.parameters():
#     if param.dtype == torch.bfloat16:
#         param.data = param.data.to(torch.float16)

## Setup For Training

2 objects:

1. A LoraConfig object with our hyperparameters for LoRA
2. An SFTConfig with our overall Training paramters

In [14]:
# LoRA Parameters

lora_parameters = LoraConfig(
    lora_alpha = LORA_ALPHA,
    lora_dropout = LORA_DROPOUT,
    r = LORA_R, 
    bias = "none", #disables training of its bias vectors in the target Linear layers, saving parameters and reducing overfit risk.
    task_type = "CAUSAL_LM", #tells PEFT that the adapter is attached to an autoregressiveLM
    target_modules = TARGET_MODULES, # applies LoRA to ATTENTION_LAYERS only (here in LITE_MODE)
)

In [15]:
# Training Paramters
train_parameters = SFTConfig(
    output_dir = PROJECT_RUN_NAME, #local directory where checkpoints, tokenizer files, and trainer states will be written.
    num_train_epochs = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size = 1, #processes validation samples individually to conserve VRAM during evaluation runs.
    gradient_accumulation_steps = GRADIENT_ACCUMULATION_STEPS,
    optim = OPTIMIZER, #paged_adamw_32bit
    save_steps = SAVE_STEPS, #triggers a checkpoint save every SAVE_STEPS iteration(e.g. 100 steps in LITE_MODE)
    save_total_limit = 10, #preventing kaggle disk space from filling up by automatically deleting older checkpoints, retaining only the 10 most recent ones
    logging_steps = LOG_STEPS,
    learning_rate = LEARNING_RATE,
    weight_decay = 0.001,
    # fp16 = not use_bf16,
    fp16 = True, # forcing fp16
    # bf16 = use_bf16,
    bf16 = False,
    max_grad_norm = 0.3, #clips exploding gradients to maximum norm of 0.3, ensuring stable weight updates in QLoRA
    max_steps = -1,
    warmup_ratio = WARMUP_RATIO,
    group_by_length = True, #groups samples of similar token length into the same batch to minimize unnecessary zero-padding, speeding up training throughput.
    lr_scheduler_type = LR_SCHEDULER_TYPE,
    report_to = "wandb" if LOG_TO_WANDB else None, #Integrates training logs into W&B without logging raw model files
    run_name = RUN_NAME,
    max_length = MAX_SEQUENCE_LENGTH,
    save_strategy = "steps",
    hub_strategy = "every_save",
    push_to_hub = True, #uploads LoRA adapter checkpoints directly to private HF repo everytime a checkpoint is saved
    hub_model_id = HUB_MODEL_NAME,
    hub_private_repo = True, #keep HF repo private
    eval_strategy = "steps",
    eval_steps = SAVE_STEPS, #runs evaluation on the validation dataset every save_steps.
    dataset_text_field = "text"
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


## Now, Creating Trainer

In [16]:
print(f"Using bf16: {use_bf16}")

Using bf16: False


In [17]:
fine_tuning = SFTTrainer(
    model = base_model,
    train_dataset = train,
    eval_dataset = val, 
    peft_config = lora_parameters, #Instructs SFTTrainer ti wrap base_model with LoRA adapter layers automatically using the configuration defined 
    args = train_parameters, #passes SFTConfig paramters(lr, optimizer, save strategy etc.)
    processing_class = tokenizer,
)

Adding EOS to train dataset:   0%|          | 0/20000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/20000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/20000 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

In [18]:
# Fine-Tune loop
fine_tuning.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128001}.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: The AccumulateGrad node's stream does not match the stream of the node that produced the incoming gradient. This may incur unnecessary synchronization and break CUDA graph capture

Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
100,1.303884,1.288921,2.577013,330223.000000,0.756500
200,1.265197,1.271225,2.702768,659017.000000,0.759500
300,1.283380,1.274911,2.688985,988895.000000,0.763000
400,1.249716,1.255311,2.661730,1319755.000000,0.763000
500,1.238611,1.253647,2.666133,1649501.000000,0.760500
600,1.267544,1.247891,2.673411,1978536.000000,0.761000


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

TrainOutput(global_step=625, training_loss=1.2819684226989747, metrics={'train_runtime': 3976.1037, 'train_samples_per_second': 5.03, 'train_steps_per_second': 0.157, 'total_flos': 3.5281145503481856e+16, 'train_loss': 1.2819684226989747, 'epoch': 1.0})

In [20]:
# Push our fine-tuned model to HuggingFace(Only LoRA adapter weights ~20 to 100 MB not total model of 6+ GB)
# fine_tuning.push_to_hub(PROJECT_RUN_NAME, private=True)
fine_tuning.push_to_hub(commit_message=f"Upload {PROJECT_RUN_NAME}")
print(f"Saved to the hub: {PROJECT_RUN_NAME}")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


Saved to the hub: price-2026-08-07_14.53.02-lite


In [21]:
if LOG_TO_WANDB:
    wandb.finish() 
# closes W&B logging connection and flushes any pending metric plots(loss, lr, hardware utilization) to our online dashboard

eval/entropy,▁█▇▆▆▆
eval/loss,█▅▆▂▂▁
eval/mean_token_accuracy,▁▄██▅▆
eval/num_tokens,▁▂▄▅▇█
eval/runtime,▅▅▁▄█▅
eval/samples_per_second,▄▄█▅▁▄
eval/steps_per_second,▄▄█▅▁▄
train/entropy,██▃▂▂▂▃▁▃▄▆▆▃▄▄▇▄▄▅▅▆▆▄▅▅▆▇▆▂▃▆▅▆▇▄▄▄▆▆▅
train/epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇████
train/global_step,▁▁▁▂▂▂▂▂▂▂▂▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇███
+5,...
